<a href="https://colab.research.google.com/github/Mogi21/ml-algorithms-from-scratch/blob/main/02_unsupervised_and_advanced_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

class EigenfacesSVD:
    """
    Performs Singular Value Decomposition (SVD) on image data to extract principal components (eigenfaces).
    Includes methods for rank-k approximation and reconstruction error analysis.
    """
    def __init__(self):
        self.mean_face = None
        self.U = None
        self.S = None
        self.Vt = None

    def fit(self, X):
        """Computes the SVD of the mean-centered data."""
        self.mean_face = np.mean(X, axis=1, keepdims=True)
        X_centered = X - self.mean_face
        self.U, self.S, self.Vt = np.linalg.svd(X_centered, full_matrices=False)

    def rank_k_approximation(self, k):
        """Generates the rank-k approximation of X using the top k eigenfaces."""
        X_reconstructed = np.zeros((self.U.shape[0], self.Vt.shape[1]))
        for i in range(k):
            X_reconstructed += np.outer(self.U[:, i], self.Vt[i, :]) * self.S[i]

        X_reconstructed += self.mean_face
        return X_reconstructed

    def reconstruct_dataset(self, X, k):
        """Reconstructs a dataset using the top k eigenfaces by projecting it onto the eigenface base."""
        X_centered = X - self.mean_face
        U_k = self.U[:, :k]
        weights = U_k.T @ X_centered
        return (U_k @ weights) + self.mean_face

    def cumulative_explained_variance(self):
        """Computes the cumulative explained variance ratio of the singular values."""
        variance = self.S**2
        cumulative_variance = np.cumsum(variance)
        return cumulative_variance / np.sum(variance)

    def reconstruction_error(self, k):
        """Computes the Mean Squared Error (MSE) of the rank-k approximation."""
        m = self.Vt.shape[0]
        U_m = self.U[:, :m]

        X_centered = U_m @ np.diag(self.S) @ self.Vt
        X_full = X_centered + self.mean_face
        X_approx = self.rank_k_approximation(k)

        return np.mean((X_full - X_approx)**2)

In [2]:
def rbf_kernel(X, Y, gamma=1.0):
    """Computes the Radial Basis Function (Gaussian) kernel matrix."""
    X_sq_norm = np.sum(X**2, axis=1)
    Y_sq_norm = np.sum(Y**2, axis=1)
    dist_sq = X_sq_norm[:, np.newaxis] - 2 * (X @ Y.T) + Y_sq_norm[np.newaxis, :]
    return np.exp(-gamma * dist_sq)

def kmeans(X, K, max_iter=100, random_state=None):
    """Custom implementation of Lloyd's K-means clustering algorithm."""
    rng = np.random.default_rng(random_state)
    n = X.shape[0]

    # Initialize centres by picking K random data points
    indices = rng.choice(n, size=K, replace=False)
    centres = X[indices].copy()
    labels = np.zeros(n, dtype=int)

    for _ in range(max_iter):
        X_sq_norm = np.sum(X**2, axis=1)
        centres_sq_norm = np.sum(centres**2, axis=1)
        distances_sq = X_sq_norm[:, np.newaxis] - 2 * (X @ centres.T) + centres_sq_norm[np.newaxis, :]
        new_labels = np.argmin(distances_sq, axis=1)

        # Update centres
        for k in range(K):
            points_in_cluster = X[new_labels == k]
            if len(points_in_cluster) > 0:
                centres[k] = np.mean(points_in_cluster, axis=0)
            else:
                centres[k] = X[rng.choice(n)]

        if np.all(new_labels == labels):
            break
        labels = new_labels

    return centres, labels

In [3]:
class RBFNetwork:
    """
    RBF network for binary classification.
    Uses K-means to select representative centres and pseudo-inverse to fit weights.
    """
    def __init__(self, n_centres=10, gamma=1.0, random_state=None):
        self.n_centres = n_centres
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y):
        # 1. Find centres with K-means
        self.centres_, _ = kmeans(X, self.n_centres, random_state=self.random_state)

        # 2. Build design matrix Phi using the RBF kernel
        K_rbf = rbf_kernel(X, self.centres_, gamma=self.gamma)
        bias_column = np.ones((X.shape[0], 1))
        Phi = np.hstack((K_rbf, bias_column))

        # 3. Compute weights via Moore-Penrose pseudo-inverse
        self.weights_ = np.linalg.pinv(Phi) @ y
        return self

    def decision_function(self, X):
        K_rbf = rbf_kernel(X, self.centres_, gamma=self.gamma)
        bias_column = np.ones((X.shape[0], 1))
        Phi = np.hstack((K_rbf, bias_column))
        return Phi @ self.weights_

    def predict(self, X):
        """Predict class labels in {-1, +1}."""
        return np.sign(self.decision_function(X))

In [4]:
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_lfw_people, make_moons
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# --- 1. SVD / Eigenfaces Evaluation ---
print("[ Analyzing Facial Image Data via SVD ]")
lfw = fetch_lfw_people(min_faces_per_person=40, resize=1.0)

# Filter for a balanced dataset and transpose to match (n_features, n_samples)
X_balanced = [lfw.data[np.where(lfw.target == t)[0][:40]] for t in np.unique(lfw.target)]
X_data = np.vstack(X_balanced)
X_faces = X_data.T

ef_svd = EigenfacesSVD()
ef_svd.fit(X_faces)
explained_var = ef_svd.cumulative_explained_variance()
print(f"Extracted {X_faces.shape[0]} feature dimensions across {X_faces.shape[1]} samples.")
print(f"Variance explained by top 50 components: {explained_var[49]:.2%}")
print(f"Reconstruction MSE at k=50: {ef_svd.reconstruction_error(50):.4f}\n")

# --- 2. RBF Network vs SVM Evaluation ---
print("[ Classifying Non-Linear Data (Moons): Custom RBF Network vs Sklearn SVM ]")
X_moon, y_moon = make_moons(n_samples=300, noise=0.2, random_state=67)
y_moon = 2 * y_moon - 1 # Map to {-1, 1}

X_train, y_train = X_moon[:200], y_moon[:200]
X_test, y_test = X_moon[200:], y_moon[200:]

# Custom RBF
rbf_net = RBFNetwork(n_centres=20, gamma=5.0, random_state=69)
rbf_net.fit(X_train, y_train)
print(f"Custom RBF Network Accuracy: {np.mean(rbf_net.predict(X_test) == y_test):.2%}")

# Sklearn baseline SVM
svm = SVC(kernel="rbf", gamma=5.0, C=1.0)
svm.fit(X_train, y_train)
print(f"Sklearn SVM (RBF) Accuracy:  {np.mean(svm.predict(X_test) == y_test):.2%}")

[ Analyzing Facial Image Data via SVD ]
Extracted 11750 feature dimensions across 760 samples.
Variance explained by top 50 components: 83.54%
Reconstruction MSE at k=50: 0.0043

[ Classifying Non-Linear Data (Moons): Custom RBF Network vs Sklearn SVM ]
Custom RBF Network Accuracy: 96.00%
Sklearn SVM (RBF) Accuracy:  94.00%
